In [1]:
!cd .. && pip install .

Processing /home/jovyan/work
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for massi: filename=massi-0.1.0-py3-none-any.whl size=10139 sha256=5da75837092f1770eeacc8cf7aa004474ceecbbf0b1120a34eb254f377a8110d
  Stored in directory: /tmp/pip-ephem-wheel-cache-gbkg39dp/wheels/b3/ef/92/093562baef6fce9b92cfc59927bf9e82e306ed148e873389c0
Successfully built massi
  Attempting uninstall: massi
    Found existing installation: massi 0.1.0
    Uninstalling massi-0.1.0:
      Successfully uninstalled massi-0.1.0


1. **We start with a partial metadata file:**

In [2]:
import json
with open("base_metadata.json", "w") as bmf:
    json.dump(
        {
            # "entry_id": "miagis-demo-flood-risk-map",
            'entry_version': 1, 
            'products': ['Map 1 Potential hot-spot map for flood risk', 'Netherlands map', 'Risk_model_0_100'],
            # 'description': 'Flood-risk map of the Netherlands, 2026'
        },
        bmf)

2. **We then use the Metadata Assistant to complete the file** 

In [3]:
from massi.agents.massi import MassiScenarioFactory, Config

In [4]:
config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
factory = MassiScenarioFactory(config)

In [5]:
graph = factory.base_metadata_graph()

In [6]:
import grandalf

graph.get_graph().print_ascii()

                      +-----------+              
                      | __start__ |              
                      +-----------+              
                            *                    
                            *                    
                            *                    
            +-------------------------------+    
            | ensure_metadata_file_presence |    
            +-------------------------------+    
                            *                    
                            *                    
                            *                    
          +-----------------------------------+  
          | select_working_base_metadata_file |  
          +-----------------------------------+  
                            *                    
                            *                    
                            *                    
              +---------------------------+      
              | check_metadata_completion |      


In [7]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [8]:
import pprint

In [9]:
def stream_until_iterruption(graph, *args):
    for event in graph.stream(*args, stream_mode="values"):
        print("\n==== new event ====")
        for k, v in event.items():
            print(f"---- {k} ----")
            pprint.pp(v, indent=4)

> **_Now we begin the interaction_**

In [10]:
# Run the graph until the first interruption
stream_until_iterruption(graph, {}, thread)


==== new event ====
---- messages ----
[]
---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']

==== new event ====
---- messages ----
[]
---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']
---- selected_base_metadata_file ----
'/home/jovyan/work/_demo/base_metadata.json'

==== new event ====
---- messages ----
[]
---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']
---- selected_base_metadata_file ----
'/home/jovyan/work/_demo/base_metadata.json'
---- schema_validation_errors ----
[   {   'message': "'entry_id' is a required property",
        'encountered_value': [   'entry_version',
                                 'entry_id',
                                 'description',
                                 'products'],
        'json_path': '$',
        'failed_key': 'required'},
    {   'message': "'description' is a required property",
        'encountered_value': [   'entry_version',
                        

ConnectError: [Errno 111] Connection refused

> **_After the first interrupt -- first user query is ready_**

In [11]:
state = graph.get_state(thread)
pprint.pp(state.values.keys())
print("~~~~")
for key in state.values.keys():
    print(f"\n---- {key} ----")
    pprint.pp(state.values[key], indent=4)

dict_keys(['messages', 'base_metadata_files', 'selected_base_metadata_file', 'schema_validation_errors', 'original_metadata'])
~~~~

---- messages ----
[]

---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']

---- selected_base_metadata_file ----
'/home/jovyan/work/_demo/base_metadata.json'

---- schema_validation_errors ----
[   {   'message': "'entry_id' is a required property",
        'encountered_value': [   'entry_version',
                                 'entry_id',
                                 'description',
                                 'products'],
        'json_path': '$',
        'failed_key': 'required'},
    {   'message': "'description' is a required property",
        'encountered_value': [   'entry_version',
                                 'entry_id',
                                 'description',
                                 'products'],
        'json_path': '$',
        'failed_key': 'required'}]

---- original_metadata ----
Met

In [12]:
pprint.pp(state.values["query_plan"], indent=4)
print("~~~~")
pprint.pp(state.values["query_plan_completed"], indent=4)
print("~~~~")
pprint.pp(state.values["current_step_pointer"], indent=4)
print("~~~~")
pprint.pp(state.values["current_query_prompt"], indent=4)

KeyError: 'query_plan'

> **_After the first interrupt - will continue with prompting the user_**
>
> - make sure to enter a value

In [15]:
from unittest.mock import patch # to bypass the user input

DEFAULT_ID="test-miagis-application-for-flood-risk-mapping"
IS_MOCK_RUN=False

# if the formed query aimed at the entry_id, let's patch it with a default
if "entry_id" in state.values["current_query_prompt"].json_path and IS_MOCK_RUN: 
    # return a default value if `input` is called.
    with patch('builtins.input', return_value=DEFAULT_ID):
        stream_until_iterruption(graph, None, thread)
else: 
    stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

Please provide a unique identifier for this data entry. test-flood-risk-miagis-dataset



==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

> **_After second interrupt - user response given but not registed_**

In [16]:
print(graph.get_state(thread).values["current_query_response"])
print(graph.get_state(thread).values["selected_base_metadata_file"])

test-flood-risk-miagis-dataset
/home/jovyan/work/_demo/base_metadata.json


> **_After the second interrupt -- metadata will now be updated_**

In [17]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

In [18]:
graph.get_state(thread).values

{'messages': [AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eval_count': 180, 'prompt_eval_du

In [19]:
keys = ["original_metadata", "updated_metadata"]
for key in keys:
    print(f"\n---- {key} ----")
    pprint.pp(graph.get_state(thread).values[key], indent=4)


---- original_metadata ----
MetadataUpdate(metadata={'entry_version': 1, 'products': ['Map 1 Potential hot-spot map for flood risk', 'Netherlands map', 'Risk_model_0_100']})

---- updated_metadata ----
MetadataUpdate(metadata={'entry_version': 1, 'products': ['Map 1 Potential hot-spot map for flood risk', 'Netherlands map', 'Risk_model_0_100'], 'entry_id': 'test-flood-risk-miagis-dataset', 'description': '<user-provided response>'})


> **_Now we can check the result of the metadata update_**

In [20]:
import json
with open(graph.get_state(thread).values["selected_base_metadata_file"], "r") as bmf:
    base_metadata = json.load(bmf)
    print(base_metadata)

{'entry_version': 1, 'products': ['Map 1 Potential hot-spot map for flood risk', 'Netherlands map', 'Risk_model_0_100'], 'entry_id': 'test-flood-risk-miagis-dataset', 'description': '<user-provided response>'}


In [21]:
assert base_metadata["entry_id"] == DEFAULT_ID if IS_MOCK_RUN else ...

> **_After the second interrupt - will continue to the next query step (because two values are missing)_**

In [22]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

In [24]:
graph.get_state(thread).values["current_query_prompt"]

QueryStep(id=2, key='data_description', json_path='$.description', priority=5, query='Please provide a description of the GIS data, including its purpose and scope.')

> **_After the third interrupt - will prompt the next query step (because two values are missing)_**
>
> - don't forget to provide an input

In [25]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

Please provide a description of the GIS data, including its purpose and scope. A map depicting flood risk areas in the Netherlands



==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

In [30]:
print(f"{graph.get_state(thread).values["query_plan_completed"]=}")

graph.get_state(thread).values["query_plan_completed"]=True


> **_After the four interrupt -- metadata will now be updated_**

In [31]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eva

In [32]:
graph.get_state(thread).values

{'messages': [AIMessage(content="Here are the JSON schema validation errors translated into human-readable descriptions:\n\n- The `entry_id` property was not provided, but it is a required field in the MIAGIS base metadata file.\n- The `description` property was not provided, but it is also a required field in the MIAGIS base metadata file.\n\nThe error messages indicate that the following properties were encountered:\n```json\n['entry_version', 'entry_id', 'description', 'products']\n```\nHowever, the schema requires these specific properties to be present:\n\n- `entry_id` (required)\n- `description` (required)\n\nTo resolve this issue, ensure that both the `entry_id` and `description` fields are provided in the MIAGIS base metadata file.", additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-01-23T17:17:59.790222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 102853802068, 'load_duration': 105069713, 'prompt_eval_count': 180, 'prompt_eval_du

In [33]:
keys = ["original_metadata", "updated_metadata"]
for key in keys:
    print(f"\n---- {key} ----")
    pprint.pp(graph.get_state(thread).values[key], indent=4)


---- original_metadata ----
MetadataUpdate(metadata={'entry_version': 1, 'products': ['Map 1 Potential hot-spot map for flood risk', 'Netherlands map', 'Risk_model_0_100']})

---- updated_metadata ----
MetadataUpdate(metadata={'entry_version': 1, 'products': ['Map 1 Potential hot-spot map for flood risk', 'Netherlands map', 'Risk_model_0_100'], 'entry_id': 'test-flood-risk-miagis-dataset', 'description': 'A map depicting flood risk areas in the Netherlands, providing a comprehensive overview of potential high-risk zones and aiding stakeholders in making informed decisions regarding flood mitigation strategies.'})


> **_Now we can check the result of the metadata update_**

In [36]:
import json
with open(graph.get_state(thread).values["selected_base_metadata_file"], "r") as bmf:
    base_metadata = json.load(bmf)
    pprint.pp(base_metadata, indent=4)

{   'entry_version': 1,
    'products': [   'Map 1 Potential hot-spot map for flood risk',
                    'Netherlands map',
                    'Risk_model_0_100'],
    'entry_id': 'test-flood-risk-miagis-dataset',
    'description': 'A map depicting flood risk areas in the Netherlands, '
                   'providing a comprehensive overview of potential high-risk '
                   'zones and aiding stakeholders in making informed decisions '
                   'regarding flood mitigation strategies.'}
